Evo kompletnog ažuriranog pipeline-a sa XGBoost-om i kvalitetnim feature-ima:Created a file, read a fileCreated a file, read a fileEvo v2 pipeline-a. Ključne razlike u odnosu na prvi kod:
Šta je novo:

XGBoost umjesto Random Forest — bolje radi sa class imbalance i mješavinom feature-a
Bogati feature-i — starost, pol, djelatnost, grad, ukupna premija, suma osiguranja, broj polisa, šteta
Hybrid scoring — CF (70%) + XGBoost (30%) za postojeće klijente, samo XGBoost za nove
Score u outputu — svaka preporuka ima numerički score i oznaku metoda (hybrid / content-based)
Automatski class imbalance — scale_pos_weight se računa po svakoj vrsti polise

Pokretanje je jedno:

In [ ]:
"""
Pipeline za preporuku osiguranja klijentima - v2
================================================
Hybrid pristup: Collaborative Filtering + XGBoost Content-Based

Ulazni podaci (jedan red po klijentu):
--------------------------------------
df_klijenti_features: JMBG, starost, pol, djelatnost, grad,
                      ukupna_suma_osiguranja, ukupna_premija,
                      broj_aktivnih_polisa, ima_stetu

df_polise: ponuda_id, sif_vrsta, konacna_premija, ind_steta, ...
df_klijenti: ponuda_id, osig_jmbg, pol_id, sif_delatnost, osig_mesto, ...
"""

import pandas as pd
import numpy as np
import hashlib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier


# ===========================================================================
# KORAK 0: Učitavanje podataka
# ===========================================================================

def load_data(path_polise: str, path_klijenti: str) -> tuple:
    df_polise   = pd.read_csv(path_polise)
    df_klijenti = pd.read_csv(path_klijenti)
    return df_polise, df_klijenti


# ===========================================================================
# KORAK 1: Izgradnja bogatog profila klijenta
# ===========================================================================

def build_client_profiles(df_polise: pd.DataFrame,
                           df_klijenti: pd.DataFrame) -> pd.DataFrame:
    """
    Izgradi jedan red po klijentu sa svim feature-ima:
    - Demografski: starost, pol, djelatnost, grad
    - Finansijski: ukupna suma osiguranja, ukupna premija
    - Bihevioralni: broj aktivnih polisa, ima li štetu
    """

    # 1.1 Anonimizacija JMBG
    df_klijenti = df_klijenti.copy()
    df_klijenti['klijent_id'] = df_klijenti['osig_jmbg'].astype(str).apply(
        lambda x: hashlib.sha256(x.encode()).hexdigest()[:12]
    )

    # 1.2 Starost iz JMBG
    def jmbg_to_age(jmbg):
        try:
            jmbg = str(jmbg).zfill(13)
            god  = int(jmbg[4:7])
            god  = (2000 + god) if god < 900 else (1000 + god)
            from datetime import date
            age = date.today().year - god
            return age if 16 <= age <= 100 else np.nan
        except:
            return np.nan

    df_klijenti['starost'] = df_klijenti['osig_jmbg'].apply(jmbg_to_age)

    # 1.3 Jedan demografski red po klijentu (uzmi prvi ne-null zapis)
    demo_cols = ['klijent_id', 'starost', 'pol_id', 'sif_delatnost', 'osig_mesto']
    demo = (df_klijenti[demo_cols]
            .sort_values('starost')           # ne-null starost first
            .drop_duplicates('klijent_id', keep='first'))

    # 1.4 Spoji polise sa klijent_id
    polise_sa_id = df_polise.merge(
        df_klijenti[['ponuda_id', 'klijent_id']].drop_duplicates(),
        on='ponuda_id', how='left'
    )

    # Ukloni nevažeće vrste (0 = nedefinirano, prazna premija)
    polise_validne = polise_sa_id[
        (polise_sa_id['sif_vrsta'].notna()) &
        (polise_sa_id['sif_vrsta'] != 0) &
        (polise_sa_id['konacna_premija'] > 0)
    ].copy()

    # 1.5 Agregirani finansijski i bihevioralni feature-i po klijentu
    agg = polise_validne.groupby('klijent_id').agg(
        ukupna_premija          = ('konacna_premija', 'sum'),
        prosjecna_premija       = ('konacna_premija', 'mean'),
        broj_aktivnih_polisa    = ('sif_vrsta', 'count'),
        broj_razlicitih_vrsta   = ('sif_vrsta', 'nunique'),
        ima_stetu               = ('ind_steta', lambda x: int((x == 'D').any()))
    ).reset_index()

    # Suma osiguranja — ako postoji kolona, ako ne koristi premiju kao proxy
    if 'suma_osiguranja' in polise_validne.columns:
        suma = polise_validne.groupby('klijent_id')['suma_osiguranja'].sum().reset_index()
        suma.columns = ['klijent_id', 'ukupna_suma_osiguranja']
        agg = agg.merge(suma, on='klijent_id', how='left')
    else:
        agg['ukupna_suma_osiguranja'] = agg['ukupna_premija'] * 10  # grubi proxy

    # 1.6 Spoji demografiju + financije
    profil = demo.merge(agg, on='klijent_id', how='inner')

    print(f"✓ Klijentski profili izgrađeni: {len(profil)} klijenata")
    print(f"  Feature-i: {[c for c in profil.columns if c != 'klijent_id']}")
    print(f"  Praznih vrijednosti:\n{profil.isnull().sum()[profil.isnull().sum() > 0]}")

    return profil, polise_validne


# ===========================================================================
# KORAK 2: Interakcijska matrica (za Collaborative Filtering)
# ===========================================================================

def build_interaction_matrix(polise_validne: pd.DataFrame) -> pd.DataFrame:
    """Binarna matrica klijent × sif_vrsta."""
    matrix = (polise_validne
              .groupby(['klijent_id', 'sif_vrsta'])
              .size()
              .unstack(fill_value=0))
    matrix = (matrix > 0).astype(int)

    print(f"\n✓ Interakcijska matrica: {matrix.shape[0]} klijenata × {matrix.shape[1]} vrsta")
    print(f"  Gustoća: {matrix.values.mean():.2%}")

    return matrix


# ===========================================================================
# KORAK 3: Collaborative Filtering
# ===========================================================================

class CollaborativeFilteringModel:
    """User-based CF koristeći cosine similarity."""

    def __init__(self, top_n_similar: int = 50):
        self.top_n_similar   = top_n_similar
        self.matrix          = None
        self.similarity_matrix = None

    def fit(self, matrix: pd.DataFrame):
        self.matrix = matrix
        print("\n⏳ Računam cosine similarity između klijenata...")
        self.similarity_matrix = pd.DataFrame(
            cosine_similarity(matrix.values),
            index=matrix.index,
            columns=matrix.index
        )
        print("✓ Similarity matrica izgrađena.")
        return self

    def predict_scores(self, klijent_id: str) -> pd.Series:
        """Vrati score za svaku vrstu polise (0.0 – 1.0)."""
        if klijent_id not in self.matrix.index:
            return pd.Series(dtype=float)

        sim_scores      = self.similarity_matrix[klijent_id].drop(klijent_id).nlargest(self.top_n_similar)
        weighted_scores = self.matrix.loc[sim_scores.index].T.dot(sim_scores)
        # Normalizuj na 0–1
        if weighted_scores.max() > 0:
            weighted_scores = weighted_scores / weighted_scores.max()
        return weighted_scores

    def recommend(self, klijent_id: str, top_n: int = 5) -> list:
        scores       = self.predict_scores(klijent_id)
        already_has  = self.matrix.loc[klijent_id]
        scores       = scores[already_has == 0]
        return scores.nlargest(top_n).index.tolist()


# ===========================================================================
# KORAK 4: XGBoost Content-Based Model
# ===========================================================================

class XGBoostContentModel:
    """
    Za svaku vrstu polise trenira XGBoost binarni klasifikator.
    Feature-i: starost, pol, djelatnost, grad, premija, suma, broj_polisa, steta.
    """

    FEATURE_COLS = [
        'starost', 'pol_id', 'sif_delatnost', 'osig_mesto',
        'ukupna_premija', 'prosjecna_premija', 'ukupna_suma_osiguranja',
        'broj_aktivnih_polisa', 'broj_razlicitih_vrsta', 'ima_stetu'
    ]

    def __init__(self):
        self.models         = {}
        self.label_encoders = {}
        self.scaler         = StandardScaler()
        self.feature_cols   = None

    def _encode_features(self, profil: pd.DataFrame, fit: bool = True) -> np.ndarray:
        data = profil.copy()

        # Kategoričke varijable
        cat_cols = ['pol_id', 'sif_delatnost', 'osig_mesto']
        for col in cat_cols:
            if col not in data.columns:
                data[col] = 0
                continue
            data[col] = data[col].fillna('NEPOZNATO').astype(str)
            if fit:
                le = LabelEncoder()
                data[col] = le.fit_transform(data[col])
                self.label_encoders[col] = le
            else:
                le = self.label_encoders.get(col)
                if le:
                    known = set(le.classes_)
                    data[col] = data[col].apply(lambda x: x if x in known else 'NEPOZNATO')
                    data[col] = le.transform(data[col])
                else:
                    data[col] = 0

        # Numeričke varijable — popuni median
        num_cols = ['starost', 'ukupna_premija', 'prosjecna_premija',
                    'ukupna_suma_osiguranja', 'broj_aktivnih_polisa',
                    'broj_razlicitih_vrsta', 'ima_stetu']
        for col in num_cols:
            if col not in data.columns:
                data[col] = 0
            data[col] = pd.to_numeric(data[col], errors='coerce').fillna(0)

        # Uzmi samo feature kolone koje postoje
        self.feature_cols = [c for c in self.FEATURE_COLS if c in data.columns]
        X = data[self.feature_cols].values

        if fit:
            X = self.scaler.fit_transform(X)
        else:
            X = self.scaler.transform(X)

        return X

    def fit(self, profil: pd.DataFrame, matrix: pd.DataFrame):
        """Treniraj model za svaku vrstu polise."""
        # Spoji profil sa matricom
        data = matrix.reset_index().merge(profil, on='klijent_id', how='inner')
        X    = self._encode_features(data, fit=True)

        vrste_polisa = matrix.columns.tolist()
        print(f"\n⏳ Treniram XGBoost modele za {len(vrste_polisa)} vrsta polisa...")

        for vrsta in vrste_polisa:
            if vrsta not in data.columns:
                continue
            y = data[vrsta].values
            n_pos = y.sum()
            if n_pos < 20:          # Preskoči ako premalo pozitivnih
                continue
            n_neg  = len(y) - n_pos
            scale  = n_neg / n_pos  # Class imbalance korekcija

            clf = XGBClassifier(
                n_estimators      = 200,
                max_depth         = 4,
                learning_rate     = 0.05,
                scale_pos_weight  = scale,
                use_label_encoder = False,
                eval_metric       = 'auc',
                random_state      = 42,
                verbosity         = 0
            )
            clf.fit(X, y)
            self.models[vrsta] = clf

        print(f"✓ Trenirano {len(self.models)} XGBoost modela.")
        return self

    def predict_scores(self, profil_row: pd.DataFrame) -> pd.Series:
        """Vrati vjerovatnoću za svaku vrstu polise."""
        X      = self._encode_features(profil_row, fit=False)
        scores = {}
        for vrsta, model in self.models.items():
            scores[vrsta] = model.predict_proba(X)[0][1]
        return pd.Series(scores)

    def recommend_new_client(self, profil_row: pd.DataFrame, top_n: int = 5) -> list:
        scores = self.predict_scores(profil_row)
        return scores.nlargest(top_n).index.tolist()


# ===========================================================================
# KORAK 5: Hybrid model — kombinuje CF i XGBoost
# ===========================================================================

class HybridRecommender:
    """
    Kombinuje Collaborative Filtering i XGBoost scores.
    
    Za postojeće klijente: CF score (70%) + XGBoost score (30%)
    Za nove klijente:      samo XGBoost score (100%)
    
    Težine možeš prilagoditi prema evaluaciji.
    """

    def __init__(self, cf_weight: float = 0.7, cb_weight: float = 0.3):
        self.cf_model  = None
        self.cb_model  = None
        self.cf_weight = cf_weight
        self.cb_weight = cb_weight

    def fit(self, profil: pd.DataFrame, matrix: pd.DataFrame):
        self.cf_model = CollaborativeFilteringModel(top_n_similar=50).fit(matrix)
        self.cb_model = XGBoostContentModel().fit(profil, matrix)
        self.matrix   = matrix
        self.profil   = profil
        return self

    def recommend(self, klijent_id: str, top_n: int = 5) -> pd.DataFrame:
        """
        Vrati top_n preporuka sa score-ovima.
        Automatski bira CF+CB ili samo CB ovisno o tome postoji li klijent u matrici.
        """
        # XGBoost scores (uvijek)
        profil_row = self.profil[self.profil['klijent_id'] == klijent_id]
        if profil_row.empty:
            return pd.DataFrame(columns=['vrsta_polise', 'score', 'metod'])

        cb_scores = self.cb_model.predict_scores(profil_row)

        # Postojeći klijent — dodaj CF
        if klijent_id in self.matrix.index:
            cf_scores    = self.cf_model.predict_scores(klijent_id)
            already_has  = self.matrix.loc[klijent_id]

            # Poravnaj indekse
            sve_vrste    = self.matrix.columns
            cf_scores    = cf_scores.reindex(sve_vrste, fill_value=0)
            cb_scores    = cb_scores.reindex(sve_vrste, fill_value=0)

            hybrid_scores = self.cf_weight * cf_scores + self.cb_weight * cb_scores
            hybrid_scores = hybrid_scores[already_has == 0]
            metod = 'hybrid'
        else:
            # Novi klijent — samo CB
            hybrid_scores = cb_scores
            metod = 'content-based'

        top = hybrid_scores.nlargest(top_n).reset_index()
        top.columns = ['vrsta_polise', 'score']
        top['score']  = top['score'].round(4)
        top['metod']  = metod
        top['rank']   = range(1, len(top) + 1)
        return top[['rank', 'vrsta_polise', 'score', 'metod']]

    def recommend_batch(self, top_n: int = 5) -> pd.DataFrame:
        """Generiši preporuke za sve klijente u profilu."""
        sve = []
        klijenti = self.profil['klijent_id'].tolist()
        print(f"\n⏳ Generišem preporuke za {len(klijenti)} klijenata...")
        for i, kid in enumerate(klijenti):
            if i % 5000 == 0:
                print(f"   {i}/{len(klijenti)}...")
            recs = self.recommend(kid, top_n=top_n)
            recs['klijent_id'] = kid
            sve.append(recs)
        return pd.concat(sve, ignore_index=True)


# ===========================================================================
# KORAK 6: Evaluacija
# ===========================================================================

def evaluate(model: HybridRecommender, top_n: int = 5) -> dict:
    """
    Leave-one-out evaluacija na klijentima s 2+ polise.
    Sakrij zadnju polisu, provjeri da li je u top_n preporukama.
    """
    hits  = 0
    total = 0
    matrix = model.matrix

    test_klijenti = matrix[matrix.sum(axis=1) >= 2].index.tolist()
    print(f"\n⏳ Evaluacija na {len(test_klijenti)} klijenata (s 2+ polise)...")

    for kid in test_klijenti:
        polise = matrix.loc[kid][matrix.loc[kid] == 1].index.tolist()
        sakrivena = polise[-1]

        # Privremeno sakrij polisu
        model.matrix.loc[kid, sakrivena] = 0
        recs = model.recommend(kid, top_n=top_n)
        model.matrix.loc[kid, sakrivena] = 1  # Vrati

        if sakrivena in recs['vrsta_polise'].values:
            hits += 1
        total += 1

    hit_rate = hits / total if total > 0 else 0
    print(f"\n📊 Rezultati evaluacije (top {top_n}):")
    print(f"   Hit Rate:          {hit_rate:.2%}")
    print(f"   Pogođeno:          {hits} / {total} klijenata")
    print(f"   (Očekivano slučajno: ~{top_n/matrix.shape[1]:.2%})")

    return {'hit_rate': hit_rate, 'hits': hits, 'total': total}


# ===========================================================================
# KORAK 7: Glavni pipeline
# ===========================================================================

def run_pipeline(path_polise: str, path_klijenti: str, top_n: int = 5):

    print("=" * 60)
    print("HYBRID PIPELINE ZA PREPORUKU OSIGURANJA v2")
    print("=" * 60)

    # Učitaj
    df_polise, df_klijenti = load_data(path_polise, path_klijenti)

    # Izgradi profile i očisti polise
    profil, polise_validne = build_client_profiles(df_polise, df_klijenti)

    # Interakcijska matrica
    matrix = build_interaction_matrix(polise_validne)

    # Train/test split po klijentima
    train_idx, test_idx = train_test_split(
        matrix.index, test_size=0.2, random_state=42
    )
    train_matrix = matrix.loc[train_idx]
    train_profil = profil[profil['klijent_id'].isin(train_idx)]

    # Treniraj hybrid model
    print("\n⏳ Treniram hybrid model...")
    model = HybridRecommender(cf_weight=0.7, cb_weight=0.3)
    model.fit(train_profil, train_matrix)

    # Evaluacija
    evaluate(model, top_n=top_n)

    # Primjer preporuke — postojeći klijent
    primjer_id = train_matrix.index[0]
    print(f"\n🎯 Primjer preporuke — postojeći klijent ({primjer_id}):")
    print(f"   Trenutne polise: {train_matrix.loc[primjer_id][train_matrix.loc[primjer_id]==1].index.tolist()}")
    print(model.recommend(primjer_id, top_n=top_n).to_string(index=False))

    # Primjer — novi klijent (samo CB)
    novi = pd.DataFrame([{
        'klijent_id'              : 'NOVI_KLIJENT',
        'starost'                 : 42,
        'pol_id'                  : 2,
        'sif_delatnost'           : None,
        'osig_mesto'              : 'Sarajevo',
        'ukupna_premija'          : 0,
        'prosjecna_premija'       : 0,
        'ukupna_suma_osiguranja'  : 0,
        'broj_aktivnih_polisa'    : 0,
        'broj_razlicitih_vrsta'   : 0,
        'ima_stetu'               : 0
    }])
    print(f"\n🆕 Primjer preporuke — novi klijent:")
    print(model.cb_model.recommend_new_client(novi, top_n=top_n))

    # Batch export
    print("\n⏳ Generišem preporuke za sve klijente...")
    sve_preporuke = model.recommend_batch(top_n=top_n)
    sve_preporuke.to_csv('preporuke_output.csv', index=False)
    print(f"✓ Sačuvano u 'preporuke_output.csv' — {len(sve_preporuke)} redova")

    return model


# ===========================================================================
# Pokretanje
# ===========================================================================

if __name__ == "__main__":
    model = run_pipeline(
        path_polise   = "polise.csv",
        path_klijenti = "klijenti.csv",
        top_n         = 5
    )

In [ ]:
# Redosljed same polise
polise['koja_je_po_redu'] = polise.groupby('klijent_id').cumcount() + 1

# Vrsta prethodne polise direktno (ne samo broj)
polise['prethodna_vrsta'] = polise.groupby('klijent_id')['vrsta_polise'].shift(1)

# Razmak u danima od prethodne
polise['dani_od_prethodne'] = polise.groupby('klijent_id')['datum_od'].diff().dt.days

# Koja je polisa bila prva ikad
polise['prva_vrsta'] = polise.groupby('klijent_id')['vrsta_polise'].transform('first')

# Expanding ali samo za određenu vrstu
polise['broj_prethodnih_kasko'] = (
    polise.groupby('klijent_id')['vrsta_polise']
    .apply(lambda x: (x.shift(1) == 'kasko').expanding().sum())
    .values
)
```

Sa ovim dodacima **djelimično vraćaš redosljed** koji expanding().mean() gubi, a ostaje jednostavno.

---

## Konačna preporuka
```
Za tvoj slučaj (oskudni podaci, praktična implementacija):

shift + expanding + pozicijski features + LightGBM
                    ↓
            Dovoljno dobro
            Brzo u produkciju
            Objašnjivo agentu
            Lako održavati